# Schaeffer battery-system field telemetry

This notebook verifies the released archive, profiles all 28 timestamped battery systems, and shows the measured layout without assigning a fault diagnosis.

## 1. Released systems

This step lists every CSV member in the released field-data archive. One row represents one returned eight-cell LFP battery system.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'fielddata').exists())
sys.path.insert(0, str(ROOT))
from fielddata import load, systems

released = systems('schaeffer')
display(released)

,system,file,bytes
0,1,field_data/data_sys_1.csv,993816767
1,2,field_data/data_sys_2.csv,377831531
2,3,field_data/data_sys_3.csv,137687310
3,4,field_data/data_sys_4.csv,690633709
4,5,field_data/data_sys_5.csv,1454926692
5,6,field_data/data_sys_6.csv,2889184963
6,7,field_data/data_sys_7.csv,327748004
7,8,field_data/data_sys_8.csv,3622009420
8,9,field_data/data_sys_9.csv,746318877
9,10,field_data/data_sys_10.csv,608241652


The release contains 28 systems, each represented by one timestamped CSV file.

## 2. Per-system profile

The profile streams one released system at a time and writes each completed row immediately. It records the timestamp span, measured median cadence, rows, all-cell voltage range, pack current range, SOC range, and temperature range so an interrupted run can resume.

In [2]:
PROFILE_PATH = ROOT / 'reports' / 'schaeffer_system_profile.csv'
PROFILE_COLUMNS = ['system', 'record_start', 'record_end', 'duration_days', 'rows', 'median_cadence_seconds', 'cell_voltage_min_V', 'cell_voltage_max_V', 'maximum_cell_spread_mV', 'current_min_A', 'current_max_A', 'soc_min_pct', 'soc_max_pct', 'temperature_min_degC', 'temperature_max_degC']
CELL_COLUMNS = [f'U_Cell_{index}' for index in range(1, 9)]
TEMPERATURE_COLUMNS = [f'Temperature_{index}' for index in range(1, 5)]
profile = pd.read_csv(PROFILE_PATH) if PROFILE_PATH.exists() else pd.DataFrame(columns=PROFILE_COLUMNS)
done = set(profile['system'].astype(int)) if not profile.empty else set()

for system in released['system']:
    if system in done:
        continue
    start = end = previous = None
    gaps = []
    rows = 0
    cell_min, cell_max, max_spread = np.inf, -np.inf, -np.inf
    current_min, current_max = np.inf, -np.inf
    soc_min, soc_max = np.inf, -np.inf
    temp_min, temp_max = np.inf, -np.inf
    for chunk in load('schaeffer', system=int(system), chunksize=250000):
        timestamps = chunk['Timestamp']
        start = timestamps.iloc[0] if start is None else start
        end = timestamps.iloc[-1]
        chunk_gaps = timestamps.diff().dt.total_seconds().dropna().to_numpy()
        if previous is not None:
            gaps.append((timestamps.iloc[0] - previous).total_seconds())
        if len(chunk_gaps):
            gaps.extend(chunk_gaps[::max(1, len(chunk_gaps) // 2000)])
        previous = end
        rows += len(chunk)
        cells = chunk[CELL_COLUMNS].apply(pd.to_numeric, errors='coerce')
        cell_min = min(cell_min, float(cells.min().min()))
        cell_max = max(cell_max, float(cells.max().max()))
        max_spread = max(max_spread, float((cells.max(axis=1) - cells.min(axis=1)).max() * 1000))
        current = pd.to_numeric(chunk['I_Battery'], errors='coerce')
        soc = pd.to_numeric(chunk['SOC_Battery'], errors='coerce')
        temperatures = chunk[TEMPERATURE_COLUMNS].apply(pd.to_numeric, errors='coerce')
        current_min, current_max = min(current_min, float(current.min())), max(current_max, float(current.max()))
        soc_min, soc_max = min(soc_min, float(soc.min())), max(soc_max, float(soc.max()))
        temp_min, temp_max = min(temp_min, float(temperatures.min().min())), max(temp_max, float(temperatures.max().max()))
    result = {'system': int(system), 'record_start': start, 'record_end': end, 'duration_days': (end - start).total_seconds() / 86400, 'rows': rows, 'median_cadence_seconds': float(np.median(gaps)), 'cell_voltage_min_V': cell_min, 'cell_voltage_max_V': cell_max, 'maximum_cell_spread_mV': max_spread, 'current_min_A': current_min, 'current_max_A': current_max, 'soc_min_pct': soc_min, 'soc_max_pct': soc_max, 'temperature_min_degC': temp_min, 'temperature_max_degC': temp_max}
    profile = pd.concat([profile.loc[profile['system'] != system], pd.DataFrame([result])], ignore_index=True)
    profile.sort_values('system').to_csv(PROFILE_PATH, index=False)
    print(f'Completed system {system:02d}', flush=True)

profile = pd.read_csv(PROFILE_PATH).sort_values('system')
display(profile)

,system,record_start,record_end,duration_days,rows,median_cadence_seconds,cell_voltage_min_V,cell_voltage_max_V,maximum_cell_spread_mV,current_min_A,current_max_A,soc_min_pct,soc_max_pct,temperature_min_degC,temperature_max_degC
0,1,2016-02-17 15:03:03,2019-12-20 15:37:00,1402.023576,6453508,5.0,1.910,4.083,1142.0,-247.679321,8.397459e+01,0.0,100.0,-4.0,39.0
1,2,2013-10-30 07:53:05,2016-07-09 06:27:32,982.940590,2528603,5.0,0.000,3.707,1331.0,-200.854034,2.620623e+01,37.1,100.0,-2.0,30.0
2,3,2013-03-07 11:15:05,2013-08-14 11:16:00,160.000637,937123,5.0,0.110,5.949,4640.0,-1047.540649,inf,0.0,100.0,-25.0,40.0
3,4,2015-02-09 21:22:22,2016-07-22 10:21:15,528.540891,4695793,5.0,1.752,4.934,1629.0,-6696.392090,1.587994e+04,0.0,100.0,8.0,47.0
4,5,2013-03-25 05:08:05,2019-02-28 10:36:36,2166.228137,9763491,5.0,0.190,3.920,3117.0,-201.519989,3.174058e+01,25.1,100.0,-25.0,34.0
5,6,2018-04-28 18:45:05,2022-01-10 17:06:00,1352.931192,19248213,5.0,2.484,3.957,622.0,-211.720398,1.532691e+02,28.4,100.0,2.0,53.0
6,7,2015-04-18 09:58:58,2016-06-19 13:44:59,428.156956,2142191,5.0,0.259,4.203,3238.0,-201.301849,1.293733e+02,17.3,100.0,-25.0,34.0
7,8,2016-11-26 15:14:14,2022-03-03 19:04:00,1923.159560,24153489,5.0,1.979,3.824,757.0,-392.000610,1.112344e+02,4.0,100.0,0.0,84.0
8,9,2015-04-18 12:06:06,2016-10-03 15:24:24,534.137708,5001702,5.0,0.800,3.804,2271.0,-207.498657,1.241287e+02,0.0,100.0,-25.0,45.0
9,10,2015-06-04 08:09:09,2016-07-25 16:14:24,417.336979,4058574,5.0,2.336,4.013,901.0,-201.653702,1.252969e+02,0.0,100.0,-25.0,120.0


The saved profile contains one measured row for every released system. Its ranges and cadence are calculated from the timestamped CSV records rather than inferred from a single example.

## 3. Record coverage

This table summarizes the record duration, row count, and measured median timestamp gap across the 28 saved profile rows.

In [3]:
display(profile[['duration_days', 'rows', 'median_cadence_seconds']].describe())

,duration_days,rows,median_cadence_seconds
count,28.000000,2.800000e+01,28.000000
mean,741.588154,4.742038e+06,13.107143
std,510.938078,5.332869e+06,19.726537
min,160.000637,2.660670e+05,5.000000
25%,380.238976,1.457786e+06,5.000000
50%,629.250538,3.341195e+06,5.000000
75%,986.481398,5.056178e+06,5.000000
max,2166.228137,2.415349e+07,61.000000


The summary shows the variation in available record duration, sampling density, and row count among the released systems.